In [1]:
from stable_platform_matchings.optimization.optimizer import Optimizer
from stable_platform_matchings.generation.instance_generator import InstanceGenerator

In [2]:
import pickle
from pathlib import Path

In [3]:
TEXTWIDTH = 80
SIM_SIZE = 12
N_INTS = 12
SEED = 12

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../../SyntheticInstanceGenerator/data")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [4]:
with open(GRAPH_PATH, "rb") as f:
    graph = pickle.load(f)

In [5]:
ig = InstanceGenerator(
    FARMERS_PATH,
    FARMERS_14_PATH,
    INTS_PATH,
    GRAPH_PATH,
    ALPHA_PATH,
    SIGMAS_PATH
)

In [6]:
ig.gen_intermediaries(n_intermediaries=12, seed=12)
ig.gen_pickups(seed=67, scale=2, n_cycles=10)

In [7]:
platform = ig.gen_instance(
    instance_id=100,
    day=100,
    n_hist_sets=3,
    dev_mode="required_only"
)

In [8]:
ig.pickups_df

,farmer_id,farmer_x,farmer_y,farmer_lon,farmer_lat,cycle_phase,quantity,intermediary_id,nominal_day,schedule_offset,day,scaled_quantity
2,goofy_kalam_d0_f0,885867.880185,-39682.515101,102.465635,-0.358359,0,1.1,goofy_kalam,14,5,19,1.1
3,goofy_kalam_d0_f0,885867.880185,-39682.515101,102.465635,-0.358359,0,1.1,goofy_kalam,28,-4,24,1.1
5,goofy_kalam_d0_f0,885867.880185,-39682.515101,102.465635,-0.358359,0,1.1,goofy_kalam,56,1,57,1.1
6,goofy_kalam_d0_f0,885867.880185,-39682.515101,102.465635,-0.358359,0,1.1,goofy_kalam,70,0,70,1.1
7,goofy_kalam_d0_f0,885867.880185,-39682.515101,102.465635,-0.358359,0,1.1,goofy_kalam,84,-3,81,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...
7457,wonderful_jepsen_d13_f1,874617.880185,-30682.515101,102.364687,-0.277113,13,2.2,wonderful_jepsen,69,0,69,2.2
7458,wonderful_jepsen_d13_f1,874617.880185,-30682.515101,102.364687,-0.277113,13,2.2,wonderful_jepsen,83,5,88,2.2
7459,wonderful_jepsen_d13_f1,874617.880185,-30682.515101,102.364687,-0.277113,13,2.2,wonderful_jepsen,97,-1,96,1.4
7460,wonderful_jepsen_d13_f1,874617.880185,-30682.515101,102.364687,-0.277113,13,2.2,wonderful_jepsen,111,0,111,2.2


In [9]:
epsilon = {intermediary.id: 2 for intermediary in platform.intermediaries}
het_costs = {intermediary.id: (platform.dist_to_mill[intermediary.id] * 2) for intermediary in platform.intermediaries}

In [10]:
parameters = {
    "epsilon": epsilon,
    "backend": "gurobi",
    "het_costs": het_costs,
    "vrp_mode": "approximate"
}

opt = Optimizer(platform, parameters)



============================= Dominance Relations ==============================
  Number of relations        42
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('goofy_kalam', 'relaxed_turing'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('awesome_hopper', 'relaxed_turing'),
     ('awesome_hopper', 'affectionate_grothendieck'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_lalande', 'infallible_morse'),
     ('infallible_morse', 'relaxed_turing'),
     ('eloquent_hertz', 'infallible_morse'),
     ('stoic_r

In [11]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": False,
        "domination": False,
        "pay_unmatched": False,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



============================= Dominance Relations ==============================
  Number of relations        41
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'awesome_hopper'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'optimistic_lalande'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('stoic_pasteur', 'frosty_cerf'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('optimistic_lalande', 'awesome_hopper'),
     ('awesome_hopper', 'relaxed_turing'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_la

In [12]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": False,
        "domination": False,
        "pay_unmatched": True,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



============================= Dominance Relations ==============================
  Number of relations        41
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'awesome_hopper'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'optimistic_lalande'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('stoic_pasteur', 'frosty_cerf'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('optimistic_lalande', 'awesome_hopper'),
     ('awesome_hopper', 'relaxed_turing'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_la

In [12]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": True,
        "domination": False,
        "pay_unmatched": False,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



============================= Dominance Relations ==============================
  Number of relations        41
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'awesome_hopper'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'optimistic_lalande'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('stoic_pasteur', 'frosty_cerf'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('optimistic_lalande', 'awesome_hopper'),
     ('awesome_hopper', 'relaxed_turing'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_la

In [21]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": False,
        "domination": True,
        "pay_unmatched": False,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)


============================= Dominance Relations ==============================
  Number of relations        35
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('goofy_kalam', 'affectionate_grothendieck'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('silly_jemison', 'stoic_pasteur'),
     ('frosty_cerf', 'stoic_pasteur'),
     ('awesome_hopper', 'relaxed_turing'),
     ('awesome_hopper', 'affectionate_grothendieck'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_lalande', 'infallible_morse'),
     ('stoic_ramanujan', 'infallible_morse'),
     ('infallible_morse', 'affectionate_grothendieck'),
     ('silly_jemison', 'infallible_morse'),
     ('frosty_cerf', 'infallible_morse'),
     ('stoic_ramanujan',